# LeetCode 712 — Minimum ASCII Delete Sum for Two Strings bằng Python

Link đề: https://leetcode.com/problems/minimum-ascii-delete-sum-for-two-strings/ · Độ khó: **Medium** · Chủ đề: String, Dynamic Programming, Longest Common Subsequence

Skeleton LeetCode:

```python
class Solution:
    def minimumDeleteSum(self, s1: str, s2: str) -> int:
        ...
```

Ghi chú: ví dụ 1-2 lấy từ đề; các ví dụ khác do notebook tự đặt (đáp án đã kiểm chứng bằng code độc lập), cần thì đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: xóa một số ký tự khỏi `s1` và `s2` sao cho hai chuỗi còn lại **bằng nhau**, với **tổng mã ASCII của các ký tự bị xóa nhỏ nhất**.
2. Nắm các công cụ Python cần dùng: `ord`/`chr` (mã ASCII), tổng bằng `sum(...)` và `itertools.accumulate`, **dãy con (subsequence)**, tạo bảng 2 chiều đúng cách (tránh lỗi `[[0]*n]*m`), `functools.lru_cache`, và duyệt tập con bằng bitmask.
3. Đi từ **vét cạn đệ quy** -> **đệ quy có nhớ** -> **quy hoạch động 2 chiều từ dưới lên** -> **nhìn bài toán như "dãy con chung có trọng số lớn nhất" (LCS có trọng số)** -> **tối ưu bộ nhớ bằng một hàng**.
4. Hiểu hai mấu chốt: (a) hai chuỗi bằng nhau sau khi xóa nghĩa là phần còn lại là một **dãy con chung**; xóa ít nhất ASCII = giữ lại nhiều nhất ASCII, (b) trạng thái `dp[i][j]` = chi phí tối thiểu cho hai **tiền tố** `s1[:i]`, `s2[:j]`; hai ký tự cuối bằng nhau thì **được miễn phí**, khác nhau thì phải xóa một trong hai.
5. Tránh các lỗi kinh điển: đếm số ký tự thay vì tổng ASCII, quên điều kiện biên (hàng/cột 0 là tổng tiền tố), lệch chỉ số `i-1`, tham lam xóa "ký tự không có ở chuỗi kia", và tạo bảng bằng `[[0]*n]*m`.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s1` và `s2`, trả về **tổng mã ASCII nhỏ nhất của các ký tự bị xóa** để làm cho hai chuỗi **bằng nhau**.

**Ví dụ 1**

- Input: `s1 = "sea"`, `s2 = "eat"`
- Output: `231`
- Giải thích: xóa `"s"` khỏi `"sea"` (ASCII 115) và xóa `"t"` khỏi `"eat"` (116). Hai chuỗi cùng thành `"ea"`, tổng `115 + 116 = 231` là nhỏ nhất.

**Ví dụ 2**

- Input: `s1 = "delete"`, `s2 = "leet"`
- Output: `403`
- Giải thích: xóa `"dee"` khỏi `"delete"` để được `"let"`: `100 + 101 + 101`. Xóa `"e"` khỏi `"leet"` (101). Cả hai thành `"let"`, tổng `403`. Nếu đưa về `"lee"` hay `"eet"` thì được `433` hoặc `417`, lớn hơn.

**Ràng buộc:**

- `1 <= s1.length, s2.length <= 1000`
- `s1` và `s2` chỉ gồm chữ cái tiếng Anh viết thường.

Chú ý: "xóa" ở đây là xóa **các ký tự bất kỳ** (không nhất thiết liên tiếp), thứ tự các ký tự còn lại giữ nguyên. Có thể xóa hết (hai chuỗi rỗng bằng nhau, chi phí là tổng ASCII của cả hai).

In [1]:
import random
import time
import sys
from functools import lru_cache
from itertools import accumulate

examples = [
    (("sea", "eat"), 231),
    (("delete", "leet"), 403),
]
for (a, b), expected in examples:
    print(f"s1={a!r}, s2={b!r}: tổng ASCII cả hai = {sum(map(ord, a + b))} | kỳ vọng {expected}")

s1='sea', s2='eat': tổng ASCII cả hai = 627 | kỳ vọng 231
s1='delete', s2='leet': tổng ASCII cả hai = 1053 | kỳ vọng 403


## 3. Kiến thức nền

### 3.1. Mã ASCII: `ord` và `chr`

Mỗi ký tự có một **mã số** (ASCII/Unicode). `ord("a")` trả về `97`, `chr(97)` trả về `"a"`. Chữ thường `a..z` có mã `97..122`. Tổng mã của một chuỗi: `sum(ord(c) for c in s)` hay `sum(map(ord, s))`.

In [2]:
print(ord("s"), ord("e"), ord("a"), ord("t"))
print(chr(115), chr(97))
print(sum(ord(c) for c in "sea"), sum(map(ord, "eat")))
print([ord(c) for c in "abz"])

115 101 97 116
s a
313 314
[97, 98, 122]


### 3.2. Dãy con (subsequence) và "xóa ký tự để hai chuỗi bằng nhau"

**Dãy con** của một chuỗi là chuỗi thu được bằng cách **xóa một số ký tự** (có thể không xóa hay xóa hết) mà **giữ nguyên thứ tự**. `"ea"` là dãy con của `"sea"` (xóa `s`) và của `"eat"` (xóa `t`). Hai chuỗi trở nên bằng nhau sau khi xóa **khi và chỉ khi** phần còn lại là **dãy con chung** của chúng.

Khác với *chuỗi con liên tiếp* (substring): `"sa"` là dãy con của `"sea"` nhưng không phải substring.

In [3]:
def is_subsequence(t, s):
    it = iter(s)
    return all(c in it for c in t)        # mỗi ký tự của t phải xuất hiện lần lượt trong s

print(is_subsequence("ea", "sea"), is_subsequence("ea", "eat"))
print(is_subsequence("sa", "sea"), is_subsequence("ae", "sea"))

True True
True False


Dòng `c in it` dùng **iterator**: mỗi lần kiểm tra tiêu thụ phần đã duyệt, nên các ký tự được tìm theo đúng thứ tự (đây là cách viết gọn của việc dùng hai con trỏ).

### 3.3. Tổng ASCII của các tiền tố (`itertools.accumulate`)

Ta sẽ cần "tổng ASCII của `i` ký tự đầu" cho mọi `i` (chi phí xóa hết một tiền tố). `accumulate` tạo dãy tổng dồn.

In [4]:
s = "sea"
prefix = [0] + list(accumulate(map(ord, s)))
print(prefix)          # [0, 115, 216, 313]: prefix[i] = tổng ASCII của s[:i]
print(prefix[2], sum(map(ord, s[:2])))

[0, 115, 216, 313]
216 216


### 3.4. Bảng 2 chiều bằng list lồng nhau: lỗi `[[0]*n]*m`

Bảng `m x n` đúng cách: `[[0] * n for _ in range(m)]`. Viết `[[0] * n] * m` **sao chép tham chiếu**: mọi hàng là **cùng một list**, sửa một ô là đổi cả cột.

In [5]:
bad = [[0] * 3] * 2
bad[0][1] = 7
print("sai :", bad)           # cả hai hàng cùng đổi

good = [[0] * 3 for _ in range(2)]
good[0][1] = 7
print("đúng:", good)

sai : [[0, 7, 0], [0, 7, 0]]
đúng: [[0, 7, 0], [0, 0, 0]]


### 3.5. Ghi nhớ kết quả: `lru_cache`

Hàm đệ quy gọi lại cùng đối số nhiều lần: lưu kết quả bằng `lru_cache` để mỗi đối số chỉ tính một lần (mục này giống bài Fibonacci).

In [6]:
calls = 0
def paths(i, j):
    """Số đường đi từ (0,0) đến (i,j) trên lưới, chỉ đi sang phải hoặc xuống."""
    global calls
    calls += 1
    return 1 if i == 0 or j == 0 else paths(i - 1, j) + paths(i, j - 1)

print(paths(8, 8), "số lần gọi không nhớ:", calls)

calls = 0
@lru_cache(None)
def paths_memo(i, j):
    global calls
    calls += 1
    return 1 if i == 0 or j == 0 else paths_memo(i - 1, j) + paths_memo(i, j - 1)

print(paths_memo(8, 8), "số lần gọi có nhớ:", calls)

12870 số lần gọi không nhớ: 25739
12870 số lần gọi có nhớ: 80


### 3.6. Duyệt mọi tập con bằng bitmask (dùng cho oracle vét cạn)

Một chuỗi dài `m` có `2^m` dãy con. Mỗi số `mask` từ `0` đến `2^m - 1` mã hóa một tập con: bit `i` bằng `1` nghĩa là **giữ** ký tự `i`. Ta sẽ dùng cách này làm "thước đo độc lập" cho các chuỗi ngắn.

In [7]:
def all_subsequences(s):
    m = len(s)
    for mask in range(1 << m):
        yield "".join(s[i] for i in range(m) if mask >> i & 1)

print(sorted(all_subsequences("abc")))
print(len(list(all_subsequences("hello"))), "= 2^5")

['', 'a', 'ab', 'abc', 'ac', 'b', 'bc', 'c']
32 = 2^5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `s1 = "sea"`, `s2 = "eat"`: tổng ASCII `s1 = 115 + 101 + 97 = 313`, `s2 = 101 + 97 + 116 = 314`.

Ta chọn **chuỗi cuối cùng** `t` (dãy con chung) rồi xóa mọi thứ còn lại: chi phí `= (313 - ASCII(t)) + (314 - ASCII(t))`.

| `t` (dãy con chung) | ASCII(t) | chi phí xóa |
|---|---|---|
| `""` (rỗng) | 0 | `313 + 314 = 627` |
| `"e"` | 101 | `212 + 213 = 425` |
| `"a"` | 97 | `216 + 217 = 433` |
| `"ea"` | 198 | `115 + 116 = 231` |

Dãy con chung dài nhất `"ea"` cũng là dãy giữ **nhiều ASCII nhất**, nên chi phí **nhỏ nhất `231`**. Nhận xét quan trọng: chi phí `= tổng ASCII cả hai - 2 * ASCII(t)` nên **xóa ít nhất = giữ lại nhiều nhất**. (Chú ý: dãy con chung **dài nhất** theo số ký tự chưa chắc là dãy có ASCII lớn nhất; ta cần trọng số.)

Cách nghĩ theo "từng cặp ký tự cuối": xét ký tự cuối `a` của `"sea"` và `t` của `"eat"`. Chúng **khác nhau** nên ít nhất một trong hai phải bị xóa:

- xóa `a` (97) rồi giải bài toán `("se", "eat")`, hoặc
- xóa `t` (116) rồi giải bài toán `("sea", "ea")`.

Nếu hai ký tự cuối **bằng nhau** thì giữ cả hai (miễn phí) và giải bài toán nhỏ hơn. Đó là cấu trúc đệ quy.

## 5. Cách 1 — Brute Force / baseline: đệ quy trên hai chuỗi, không nhớ

**Ý tưởng.** Định nghĩa `f(i, j)` = chi phí nhỏ nhất để làm cho hai **hậu tố** `s1[i:]` và `s2[j:]` bằng nhau:

- Hết `s1` (`i == len(s1)`): phải xóa toàn bộ `s2[j:]` -> tổng ASCII `s2[j:]`.
- Hết `s2`: tương tự, tổng ASCII `s1[i:]`.
- `s1[i] == s2[j]`: giữ cả hai ký tự -> `f(i + 1, j + 1)`.
- Khác nhau: xóa `s1[i]` hoặc xóa `s2[j]`: `min(ord(s1[i]) + f(i + 1, j), ord(s2[j]) + f(i, j + 1))`.

Đáp án `f(0, 0)`. (Ở mục 10 ta chứng minh việc "gặp hai ký tự bằng nhau thì cứ giữ" là đúng.)

In [8]:
def min_delete_brute(s1, s2):
    m, n = len(s1), len(s2)
    def f(i, j):
        if i == m:
            return sum(map(ord, s2[j:]))
        if j == n:
            return sum(map(ord, s1[i:]))
        if s1[i] == s2[j]:
            return f(i + 1, j + 1)
        return min(ord(s1[i]) + f(i + 1, j), ord(s2[j]) + f(i, j + 1))
    return f(0, 0)

for (a, b), expected in examples:
    print(a, b, "->", min_delete_brute(a, b), "| kỳ vọng", expected)

sea eat -> 231 | kỳ vọng 231
delete leet -> 403 | kỳ vọng 403


In [9]:
class Solution1:
    """Cách 1: đệ quy không nhớ (chỉ dùng cho chuỗi ngắn)."""
    def minimumDeleteSum(self, s1: str, s2: str) -> int:
        m, n = len(s1), len(s2)

        def f(i, j):
            if i == m:
                return sum(map(ord, s2[j:]))
            if j == n:
                return sum(map(ord, s1[i:]))
            if s1[i] == s2[j]:
                return f(i + 1, j + 1)
            return min(ord(s1[i]) + f(i + 1, j), ord(s2[j]) + f(i, j + 1))

        return f(0, 0)

for (a, b), expected in examples:
    assert Solution1().minimumDeleteSum(a, b) == expected
print("cách 1 qua các ví dụ")

cách 1 qua các ví dụ


**Độ phức tạp.** Mỗi lời gọi (khi hai ký tự khác nhau) rẽ **hai nhánh**; độ sâu tối đa `m + n`. Số lời gọi cỡ `2^(m+n)` trong trường hợp xấu: **hàm mũ**. Space `O(m + n)` cho ngăn xếp đệ quy.

**Nút thắt.** Rất nhiều lời gọi **cùng `(i, j)`** (đi theo thứ tự "xóa trái rồi xóa phải" hay "xóa phải rồi xóa trái" tới cùng một trạng thái). Chỉ có `(m + 1)(n + 1)` cặp `(i, j)` khác nhau. Đếm thử:

In [10]:
def count_calls(s1, s2, memo):
    m, n = len(s1), len(s2)
    calls = 0
    seen = {}
    def f(i, j):
        nonlocal calls
        calls += 1
        if memo and (i, j) in seen:
            return seen[(i, j)]
        if i == m:
            r = sum(map(ord, s2[j:]))
        elif j == n:
            r = sum(map(ord, s1[i:]))
        elif s1[i] == s2[j]:
            r = f(i + 1, j + 1)
        else:
            r = min(ord(s1[i]) + f(i + 1, j), ord(s2[j]) + f(i, j + 1))
        seen[(i, j)] = r
        return r
    return f(0, 0), calls

for L in [4, 6, 8, 10, 12]:
    a, b = "abcdefghijkl"[:L], "lkjihgfedcba"[:L]          # không có ký tự chung ở cùng vị trí
    r1, c1 = count_calls(a, b, memo=False)
    r2, c2 = count_calls(a, b, memo=True)
    print(f"độ dài {L:>2}: kết quả {r1}=={r2} | gọi không nhớ = {c1:>8} | có nhớ = {c2:>4} | (L+1)^2 = {(L + 1) ** 2}")

độ dài  4: kết quả 820==820 | gọi không nhớ =      139 | có nhớ =   33 | (L+1)^2 = 25
độ dài  6: kết quả 1230==1230 | gọi không nhớ =     1847 | có nhớ =   73 | (L+1)^2 = 49
độ dài  8: kết quả 1432==1432 | gọi không nhớ =     9107 | có nhớ =  119 | (L+1)^2 = 81
độ dài 10: kết quả 1838==1838 | gọi không nhớ =    99307 | có nhớ =  179 | (L+1)^2 = 121


độ dài 12: kết quả 2244==2244 | gọi không nhớ =  1412911 | có nhớ =  255 | (L+1)^2 = 169


Số lời gọi không nhớ bùng nổ theo cấp số nhân, còn khi có nhớ chỉ khoảng `(L+1)^2`. Ghi nhớ là then chốt.

## 6. Ý tưởng thuật toán / bước đột phá

**Bất biến:** kết quả `f(i, j)` **chỉ phụ thuộc** cặp `(i, j)` (hai hậu tố còn lại), không phụ thuộc đường đã đi tới đó. Vì vậy chỉ có `(m + 1)(n + 1)` bài toán con, mỗi bài giải **một lần**: đó là **quy hoạch động (DP)**.

Công thức (dùng **tiền tố** cho tiện lập bảng): gọi `dp[i][j]` là chi phí nhỏ nhất để làm cho `s1[:i]` và `s2[:j]` bằng nhau:

```
dp[0][j] = ASCII(s2[:j])             # xóa hết s2[:j]
dp[i][0] = ASCII(s1[:i])             # xóa hết s1[:i]
nếu s1[i-1] == s2[j-1]:  dp[i][j] = dp[i-1][j-1]
ngược lại:               dp[i][j] = min( dp[i-1][j] + ord(s1[i-1]),     # xóa ký tự cuối của s1
                                          dp[i][j-1] + ord(s2[j-1]) )    # xóa ký tự cuối của s2
```

Đáp án `dp[m][n]`. Ta có bốn cách hiện thực/nhìn nhận:

- Cách 2: **top-down** (đệ quy + ghi nhớ).
- Cách 3: **bottom-up** lập bảng 2 chiều (dùng để nộp).
- Cách 4: đổi góc nhìn sang **LCS có trọng số**: `đáp án = tổng ASCII cả hai - 2 * (tổng ASCII lớn nhất của một dãy con chung)`.
- Cách 5: **tối ưu bộ nhớ**: mỗi hàng chỉ cần hàng ngay trước nó.

## 7. Các cách tiếp theo

### Cách 2 — Đệ quy có nhớ (top-down)

**Giải thích.** Giữ nguyên hàm `f(i, j)` của cách 1 nhưng đặt `@lru_cache(None)` để mỗi cặp `(i, j)` chỉ tính một lần. Chú ý: độ sâu đệ quy tối đa `m + n` (tới `2000`) vượt giới hạn mặc định `1000` của Python, nên cần `sys.setrecursionlimit`.

Demo:

In [11]:
def demo_memo(s1, s2):
    m, n = len(s1), len(s2)

    @lru_cache(None)
    def f(i, j):
        if i == m:
            return sum(map(ord, s2[j:]))
        if j == n:
            return sum(map(ord, s1[i:]))
        if s1[i] == s2[j]:
            return f(i + 1, j + 1)
        return min(ord(s1[i]) + f(i + 1, j), ord(s2[j]) + f(i, j + 1))

    ans = f(0, 0)
    print(f"{s1!r},{s2!r}: đáp án = {ans}, số trạng thái đã tính = {f.cache_info().currsize}")
    return ans

demo_memo("sea", "eat")
demo_memo("delete", "leet")

'sea','eat': đáp án = 231, số trạng thái đã tính = 12
'delete','leet': đáp án = 403, số trạng thái đã tính = 23


403

**Code nộp LeetCode (cách 2):**

In [12]:
class Solution2:
    """Cách 2: đệ quy có nhớ."""
    def minimumDeleteSum(self, s1: str, s2: str) -> int:
        m, n = len(s1), len(s2)
        sys.setrecursionlimit(max(sys.getrecursionlimit(), m + n + 100))

        @lru_cache(None)
        def f(i, j):
            if i == m:
                return sum(map(ord, s2[j:]))
            if j == n:
                return sum(map(ord, s1[i:]))
            if s1[i] == s2[j]:
                return f(i + 1, j + 1)
            return min(ord(s1[i]) + f(i + 1, j), ord(s2[j]) + f(i, j + 1))

        return f(0, 0)

for (a, b), expected in examples:
    assert Solution2().minimumDeleteSum(a, b) == expected
print("cách 2 qua các ví dụ")

cách 2 qua các ví dụ


**Độ phức tạp (cách 2).** Có `(m + 1)(n + 1)` trạng thái; mỗi trạng thái xử lý `O(1)` ngoài các phép `sum(...)` ở điều kiện biên (mỗi lần `O(m)` hoặc `O(n)`, chỉ xảy ra ở biên). **Time `O(m*n)`** (cộng một lượng nhỏ ở biên), Space `O(m*n)` cho bảng nhớ + `O(m + n)` ngăn xếp đệ quy. Nhược điểm: đệ quy sâu và hằng số lớn ở `m, n = 1000`.

### Cách 3 — Quy hoạch động từ dưới lên (bottom-up) - dùng để nộp

**Giải thích từng đoạn.**

1. **Bảng** `dp` kích thước `(m + 1) x (n + 1)`, `dp[i][j]` cho `s1[:i]`, `s2[:j]`.
2. **Hàng 0, cột 0**: `dp[0][j] = ASCII(s2[:j])`, `dp[i][0] = ASCII(s1[:i])` (tổng tiền tố).
3. Với `i` từ `1..m`, `j` từ `1..n`: so `s1[i-1]` với `s2[j-1]` (chỉ số `i-1` vì `dp` dùng độ dài tiền tố). Bằng nhau -> chéo `dp[i-1][j-1]`; khác nhau -> `min` hai cách xóa.
4. Đáp án `dp[m][n]`.

Demo bảng cho ví dụ 1:

In [13]:
def dp_table(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for j in range(1, n + 1):
        dp[0][j] = dp[0][j - 1] + ord(s2[j - 1])
    for i in range(1, m + 1):
        dp[i][0] = dp[i - 1][0] + ord(s1[i - 1])
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(dp[i - 1][j] + ord(s1[i - 1]), dp[i][j - 1] + ord(s2[j - 1]))
    return dp

s1, s2 = "sea", "eat"
dp = dp_table(s1, s2)
print("        " + "".join(f"{c!r:>6}" for c in "-" + s2))
for i, row in enumerate(dp):
    label = "-" if i == 0 else s1[i - 1]
    print(f"{label!r:>5}  " + "".join(f"{v:>6}" for v in row))
print("đáp án dp[3][3] =", dp[3][3])

           '-'   'e'   'a'   't'
  '-'       0   101   198   314
  's'     115   216   313   429
  'e'     216   115   212   328
  'a'     313   212   115   231
đáp án dp[3][3] = 231


**Code nộp LeetCode (cách 3) - cũng là code chính ở mục 9:**

In [14]:
class Solution3:
    """Cách 3: quy hoạch động bảng 2 chiều."""
    def minimumDeleteSum(self, s1: str, s2: str) -> int:
        m, n = len(s1), len(s2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for j in range(1, n + 1):
            dp[0][j] = dp[0][j - 1] + ord(s2[j - 1])
        for i in range(1, m + 1):
            dp[i][0] = dp[i - 1][0] + ord(s1[i - 1])
            for j in range(1, n + 1):
                if s1[i - 1] == s2[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1]
                else:
                    dp[i][j] = min(dp[i - 1][j] + ord(s1[i - 1]),
                                   dp[i][j - 1] + ord(s2[j - 1]))
        return dp[m][n]

for (a, b), expected in examples:
    assert Solution3().minimumDeleteSum(a, b) == expected
print("cách 3 qua các ví dụ")

cách 3 qua các ví dụ


**Độ phức tạp (cách 3).** **Time `O(m*n)`** (`10^6` ô với `m = n = 1000`, mỗi ô `O(1)`), **Space `O(m*n)`**.

### Cách 4 — Nhìn như LCS có trọng số: `đáp án = tổng - 2 * (ASCII dãy con chung lớn nhất)`

**Giải thích.** Ở mục 4 ta thấy: nếu giữ lại dãy con chung `t` thì chi phí `= ASCII(s1) + ASCII(s2) - 2 * ASCII(t)`. Vậy chỉ cần tìm dãy con chung có **tổng ASCII lớn nhất**: một biến thể của bài **Longest Common Subsequence** (LCS), nhưng mỗi ký tự khớp được cộng `ord(c)` thay vì `1`.

`w[i][j]` = tổng ASCII lớn nhất của dãy con chung của `s1[:i]` và `s2[:j]`:

```
w[0][*] = w[*][0] = 0
s1[i-1] == s2[j-1]:  w[i][j] = w[i-1][j-1] + ord(s1[i-1])
ngược lại:           w[i][j] = max(w[i-1][j], w[i][j-1])
```

Đáp án `= sum(ord) của s1 và s2 - 2 * w[m][n]`. Hai cách DP này **cùng mô hình nhưng khác góc nhìn** (một bên tối thiểu hóa phần bị xóa, một bên tối đa hóa phần giữ lại), nên cho ta cách đối chiếu hữu ích.

In [15]:
def weighted_lcs_table(s1, s2):
    m, n = len(s1), len(s2)
    w = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                w[i][j] = w[i - 1][j - 1] + ord(s1[i - 1])
            else:
                w[i][j] = max(w[i - 1][j], w[i][j - 1])
    return w

s1, s2 = "sea", "eat"
w = weighted_lcs_table(s1, s2)
for row in w:
    print(row)
total = sum(map(ord, s1 + s2))
print("tổng ASCII =", total, "| giữ lại tối đa =", w[-1][-1], "| đáp án =", total - 2 * w[-1][-1])

[0, 0, 0, 0]
[0, 0, 0, 0]
[0, 101, 101, 101]
[0, 101, 198, 198]
tổng ASCII = 627 | giữ lại tối đa = 198 | đáp án = 231


**Code nộp LeetCode (cách 4):**

In [16]:
class Solution4:
    """Cách 4: LCS có trọng số, đáp án = tổng - 2 * trọng số lớn nhất giữ lại."""
    def minimumDeleteSum(self, s1: str, s2: str) -> int:
        m, n = len(s1), len(s2)
        w = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if s1[i - 1] == s2[j - 1]:
                    w[i][j] = w[i - 1][j - 1] + ord(s1[i - 1])
                else:
                    w[i][j] = max(w[i - 1][j], w[i][j - 1])
        return sum(map(ord, s1)) + sum(map(ord, s2)) - 2 * w[m][n]

for (a, b), expected in examples:
    assert Solution4().minimumDeleteSum(a, b) == expected
print("cách 4 qua các ví dụ")

cách 4 qua các ví dụ


**Độ phức tạp (cách 4).** **Time `O(m*n)`**, **Space `O(m*n)`**; bảng chỉ chứa số không âm và cách này không cần xử lý hàng/cột 0 đặc biệt (đều `0`).

### Cách 5 — Tối ưu bộ nhớ: chỉ giữ một hàng

**Giải thích.** Để tính hàng `i` của `dp` ta chỉ cần hàng `i - 1` (cho `dp[i-1][j]` và `dp[i-1][j-1]`) và các ô vừa tính cùng hàng (`dp[i][j-1]`). Vì vậy dùng **một mảng `cur` độ dài `n + 1`** và một biến `prev_diag` để nhớ `dp[i-1][j-1]` trước khi bị ghi đè.

- Khởi tạo `cur[j] = ASCII(s2[:j])` (hàng 0).
- Với mỗi `i`: lưu `diag = cur[0]` (giá trị `dp[i-1][0]`), rồi cập nhật `cur[0] += ord(s1[i-1])` (giá trị `dp[i][0]`); với `j = 1..n`: lưu `old = cur[j]` (tức `dp[i-1][j]`), tính `cur[j]` mới, rồi `diag = old` cho bước kế.

Demo từng hàng:

In [17]:
def rolling_demo(s1, s2):
    n = len(s2)
    cur = [0] * (n + 1)
    for j in range(1, n + 1):
        cur[j] = cur[j - 1] + ord(s2[j - 1])
    print("hàng 0:", cur)
    for i in range(1, len(s1) + 1):
        diag = cur[0]                         # dp[i-1][0]
        cur[0] += ord(s1[i - 1])              # dp[i][0]
        for j in range(1, n + 1):
            old = cur[j]                      # dp[i-1][j]
            if s1[i - 1] == s2[j - 1]:
                cur[j] = diag
            else:
                cur[j] = min(old + ord(s1[i - 1]), cur[j - 1] + ord(s2[j - 1]))
            diag = old
        print(f"hàng {i}:", cur)
    return cur[n]

print("đáp án:", rolling_demo("sea", "eat"))

hàng 0: [0, 101, 198, 314]
hàng 1: [115, 216, 313, 429]
hàng 2: [216, 115, 212, 328]
hàng 3: [313, 212, 115, 231]
đáp án: 231


So với bảng đầy đủ của cách 3: các hàng in ra trùng từng hàng của `dp`. **Code nộp LeetCode (cách 5):**

In [18]:
class Solution5:
    """Cách 5: quy hoạch động một hàng."""
    def minimumDeleteSum(self, s1: str, s2: str) -> int:
        n = len(s2)
        cur = [0] * (n + 1)
        for j in range(1, n + 1):
            cur[j] = cur[j - 1] + ord(s2[j - 1])
        for i in range(1, len(s1) + 1):
            diag = cur[0]
            cur[0] += ord(s1[i - 1])
            for j in range(1, n + 1):
                old = cur[j]
                if s1[i - 1] == s2[j - 1]:
                    cur[j] = diag
                else:
                    cur[j] = min(old + ord(s1[i - 1]), cur[j - 1] + ord(s2[j - 1]))
                diag = old
        return cur[n]

for (a, b), expected in examples:
    assert Solution5().minimumDeleteSum(a, b) == expected
print("cách 5 qua các ví dụ")

cách 5 qua các ví dụ


**Độ phức tạp (cách 5).** **Time `O(m*n)`**, **Space `O(n)`** (có thể chọn `s2` là chuỗi ngắn hơn để bộ nhớ nhỏ hơn: bài toán đối xứng theo hai chuỗi). Đánh đổi: không còn bảng đầy đủ nên không truy vết được dãy con giữ lại (xem biến thể 15.1).

**Tóm tắt năm cách:** (1) đệ quy mũ; (2) đệ quy có nhớ; (3) bảng DP `dp[i][j]` tối thiểu hóa phần xóa; (4) LCS trọng số (tối đa hóa phần giữ); (5) DP một hàng.

## 8. TRACE từng bước

### 8.1. Trace cách 3 (bảng DP) trên `"sea"`, `"eat"`

Hàm in từng ô được điền: hai ký tự đang so, hai khả năng (xóa trái/xóa phải) hoặc ô chéo, và giá trị chọn.

| Cột | Ý nghĩa |
|---|---|
| `(i,j)` | ô đang điền, ứng với `s1[:i]`, `s2[:j]` |
| `s1[i-1], s2[j-1]` | hai ký tự cuối của hai tiền tố |
| `xóa s1` | `dp[i-1][j] + ord(s1[i-1])` |
| `xóa s2` | `dp[i][j-1] + ord(s2[j-1])` |

In [19]:
def trace_dp(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for j in range(1, n + 1):
        dp[0][j] = dp[0][j - 1] + ord(s2[j - 1])
    for i in range(1, m + 1):
        dp[i][0] = dp[i - 1][0] + ord(s1[i - 1])
    print("hàng 0:", dp[0], "| cột 0:", [dp[i][0] for i in range(m + 1)])
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            a, b = s1[i - 1], s2[j - 1]
            if a == b:
                dp[i][j] = dp[i - 1][j - 1]
                print(f"({i},{j}) {a!r} == {b!r}: giữ cả hai -> dp[{i - 1}][{j - 1}] = {dp[i][j]}")
            else:
                del1 = dp[i - 1][j] + ord(a)
                del2 = dp[i][j - 1] + ord(b)
                dp[i][j] = min(del1, del2)
                print(f"({i},{j}) {a!r} != {b!r}: xóa s1 = {del1}, xóa s2 = {del2} -> {dp[i][j]}")
    return dp[m][n]

print("đáp án:", trace_dp("sea", "eat"))

hàng 0: [0, 101, 198, 314] | cột 0: [0, 115, 216, 313]
(1,1) 's' != 'e': xóa s1 = 216, xóa s2 = 216 -> 216
(1,2) 's' != 'a': xóa s1 = 313, xóa s2 = 313 -> 313
(1,3) 's' != 't': xóa s1 = 429, xóa s2 = 429 -> 429
(2,1) 'e' == 'e': giữ cả hai -> dp[1][0] = 115
(2,2) 'e' != 'a': xóa s1 = 414, xóa s2 = 212 -> 212
(2,3) 'e' != 't': xóa s1 = 530, xóa s2 = 328 -> 328
(3,1) 'a' != 'e': xóa s1 = 212, xóa s2 = 414 -> 212
(3,2) 'a' == 'a': giữ cả hai -> dp[2][1] = 115
(3,3) 'a' != 't': xóa s1 = 425, xóa s2 = 231 -> 231
đáp án: 231


Truy ngược từ `dp[3][3] = 231`: ô `(3,3)` có `a != t` và đạt được bằng `xóa s2` (`dp[3][2] + 116 = 115 + 116`); `dp[3][2]` có `a == a` nên lấy chéo `dp[2][1]`; `dp[2][1]` có `e == e` nên lấy chéo `dp[1][0] = 115`. Vậy xóa `s` (115) khỏi `s1` và `t` (116) khỏi `s2`: `231`.

### 8.2. Trace cách 2 (đệ quy có nhớ) trên `"delete"`, `"leet"`

Thụt lề theo độ sâu, đánh dấu lúc trúng bảng nhớ (ví dụ này nhỏ nên chỉ thấy vài lần trúng).

In [20]:
def trace_memo(s1, s2):
    m, n = len(s1), len(s2)
    memo = {}
    def f(i, j, depth):
        pad = "  " * depth
        if (i, j) in memo:
            print(f"{pad}f({i},{j}) TRÚNG NHỚ = {memo[(i, j)]}")
            return memo[(i, j)]
        if i == m:
            r = sum(map(ord, s2[j:]))
            print(f"{pad}f({i},{j}) hết s1 -> xóa hết {s2[j:]!r} = {r}")
        elif j == n:
            r = sum(map(ord, s1[i:]))
            print(f"{pad}f({i},{j}) hết s2 -> xóa hết {s1[i:]!r} = {r}")
        elif s1[i] == s2[j]:
            print(f"{pad}f({i},{j}) {s1[i]!r}=={s2[j]!r}: giữ")
            r = f(i + 1, j + 1, depth + 1)
        else:
            print(f"{pad}f({i},{j}) {s1[i]!r}!={s2[j]!r}: thử xóa {s1[i]!r}({ord(s1[i])}) hoặc {s2[j]!r}({ord(s2[j])})")
            r = min(ord(s1[i]) + f(i + 1, j, depth + 1), ord(s2[j]) + f(i, j + 1, depth + 1))
        memo[(i, j)] = r
        print(f"{pad}=> f({i},{j}) = {r}")
        return r
    return f(0, 0, 0)

print("kết quả:", trace_memo("delete", "leet"))

f(0,0) 'd'!='l': thử xóa 'd'(100) hoặc 'l'(108)
  f(1,0) 'e'!='l': thử xóa 'e'(101) hoặc 'l'(108)
    f(2,0) 'l'=='l': giữ
      f(3,1) 'e'=='e': giữ
        f(4,2) 't'!='e': thử xóa 't'(116) hoặc 'e'(101)
          f(5,2) 'e'=='e': giữ
            f(6,3) hết s1 -> xóa hết 't' = 116
            => f(6,3) = 116
          => f(5,2) = 116
          f(4,3) 't'=='t': giữ
            f(5,4) hết s2 -> xóa hết 'e' = 101
            => f(5,4) = 101
          => f(4,3) = 101
        => f(4,2) = 202
      => f(3,1) = 202
    => f(2,0) = 202
    f(1,1) 'e'=='e': giữ
      f(2,2) 'l'!='e': thử xóa 'l'(108) hoặc 'e'(101)
        f(3,2) 'e'=='e': giữ
          f(4,3) TRÚNG NHỚ = 101
        => f(3,2) = 101
        f(2,3) 'l'!='t': thử xóa 'l'(108) hoặc 't'(116)
          f(3,3) 'e'!='t': thử xóa 'e'(101) hoặc 't'(116)
            f(4,3) TRÚNG NHỚ = 101
            f(3,4) hết s2 -> xóa hết 'ete' = 318
            => f(3,4) = 318
          => f(3,3) = 202
          f(2,4) hết s2 -> xóa hết 'lete' = 426

Đường tối ưu: xóa `d`, `e` (ký tự thứ hai của `delete`), rồi khớp `l`, `e`, xóa `e` còn lại của `delete`, khớp `t`, và xóa một `e` của `leet`: tổng `100 + 101 + 101 + 101 = 403`.

### 8.3. Trace cách 4 (LCS có trọng số): ô nào "giữ" ký tự nào

In ô khớp và trọng số cộng thêm.

In [21]:
def trace_wlcs(s1, s2):
    m, n = len(s1), len(s2)
    w = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                w[i][j] = w[i - 1][j - 1] + ord(s1[i - 1])
                print(f"({i},{j}) khớp {s1[i - 1]!r}: w = w[{i - 1}][{j - 1}] + {ord(s1[i - 1])} = {w[i][j]}")
            else:
                w[i][j] = max(w[i - 1][j], w[i][j - 1])
    total = sum(map(ord, s1 + s2))
    print(f"giữ tối đa {w[m][n]}; tổng ASCII {total}; đáp án = {total} - 2*{w[m][n]} = {total - 2 * w[m][n]}")

trace_wlcs("delete", "leet")

(2,2) khớp 'e': w = w[1][1] + 101 = 101
(2,3) khớp 'e': w = w[1][2] + 101 = 101
(3,1) khớp 'l': w = w[2][0] + 108 = 108
(4,2) khớp 'e': w = w[3][1] + 101 = 209
(4,3) khớp 'e': w = w[3][2] + 101 = 209
(5,4) khớp 't': w = w[4][3] + 116 = 325
(6,2) khớp 'e': w = w[5][1] + 101 = 209
(6,3) khớp 'e': w = w[5][2] + 101 = 310
giữ tối đa 325; tổng ASCII 1053; đáp án = 1053 - 2*325 = 403


Các ô khớp tạo thành dãy con chung `"let"` (`108 + 101 + 116 = 325`); hàm trên tính `tổng - 2 * 325` và ra `403`.

## 9. Code nộp LeetCode

Dùng **cách 3** (bảng DP `dp[i][j]`):

```python
class Solution:
    def minimumDeleteSum(self, s1: str, s2: str) -> int:
        m, n = len(s1), len(s2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for j in range(1, n + 1):
            dp[0][j] = dp[0][j - 1] + ord(s2[j - 1])
        for i in range(1, m + 1):
            dp[i][0] = dp[i - 1][0] + ord(s1[i - 1])
            for j in range(1, n + 1):
                if s1[i - 1] == s2[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1]
                else:
                    dp[i][j] = min(dp[i - 1][j] + ord(s1[i - 1]),
                                   dp[i][j - 1] + ord(s2[j - 1]))
        return dp[m][n]
```

Chạy lại đúng đoạn code trên:

In [22]:
class Solution:
    def minimumDeleteSum(self, s1: str, s2: str) -> int:
        m, n = len(s1), len(s2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for j in range(1, n + 1):
            dp[0][j] = dp[0][j - 1] + ord(s2[j - 1])
        for i in range(1, m + 1):
            dp[i][0] = dp[i - 1][0] + ord(s1[i - 1])
            for j in range(1, n + 1):
                if s1[i - 1] == s2[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1]
                else:
                    dp[i][j] = min(dp[i - 1][j] + ord(s1[i - 1]),
                                   dp[i][j - 1] + ord(s2[j - 1]))
        return dp[m][n]

# Mô phỏng cách LeetCode gọi
for (a, b), expected in examples:
    got = Solution().minimumDeleteSum(a, b)
    print(repr(a), repr(b), "->", got, "OK" if got == expected else "SAI")
    assert got == expected

'sea' 'eat' -> 231 OK
'delete' 'leet' -> 403 OK


## 10. Vì sao thuật toán đúng

**Bước 1: phương án cuối cùng là một dãy con chung.** Sau khi xóa, hai chuỗi bằng nhau thành `t`; `t` là dãy con của cả `s1` và `s2`. Ngược lại, với mọi dãy con chung `t`, ta có thể đạt `t` bằng cách xóa phần còn lại. Chi phí `= ASCII(s1) + ASCII(s2) - 2 * ASCII(t)`. Vậy bài toán **tương đương** tìm dãy con chung có tổng ASCII lớn nhất (cách 4 giải thẳng bài đó).

**Bước 2: công thức tiền tố.** Xét `dp[i][j]`. Nhìn ký tự cuối `a = s1[i-1]`, `b = s2[j-1]`:

- Trong phương án tối ưu cho `(i, j)`, ký tự `a` hoặc **bị xóa**, hoặc **được giữ** (khớp với một ký tự nào đó của `s2[:j]`). Tương tự `b`.
- Nếu `a` bị xóa: chi phí `ord(a)` cộng chi phí tối ưu của `(i-1, j)`. Nếu `b` bị xóa: `ord(b) + dp[i][j-1]`. Nếu cả hai được giữ: chúng phải **khớp với nhau** ở cuối chuỗi còn lại (vì là ký tự cuối của cả hai chuỗi bằng nhau), tức `a == b`, và chi phí là `dp[i-1][j-1]`.
- Khi `a == b`, ta **luôn giữ chúng** (lấy `dp[i-1][j-1]`) mà không mất tối ưu. Lý do (lập luận trao đổi): giả sử phương án tối ưu xóa `a`. Khi đó `b` hoặc cũng bị xóa (có thể giữ cả hai, tiết kiệm `ord(a) + ord(b)` mà vẫn hợp lệ, mâu thuẫn tối ưu), hoặc `b` được khớp với một ký tự `a'` của `s1[:i-1]` có cùng chữ cái; ta đổi cách ghép: khớp `b` với `a` thay vì `a'` và xóa `a'` thay vì `a`: chi phí **không đổi** (cùng chữ cái, cùng mã) và thứ tự vẫn hợp lệ (vì `a` đứng sau `a'`). Vậy luôn có phương án tối ưu giữ cặp cuối bằng nhau.
- Khi `a != b`, không thể giữ cả hai (chúng không khớp được với nhau), nên ít nhất một bị xóa; lấy `min` của hai khả năng là đủ.

**Bước 3: điều kiện biên.** `i = 0` (hết `s1`): mọi ký tự `s2[:j]` phải bị xóa: `ASCII(s2[:j])`. Tương tự `j = 0`.

**Bước 4: thứ tự điền.** `dp[i][j]` phụ thuộc `dp[i-1][j]`, `dp[i][j-1]`, `dp[i-1][j-1]`, đều đã tính nếu ta duyệt `i`, `j` tăng dần.

**Cách 5** chỉ cắt bớt bộ nhớ, giá trị y nguyên. **Cách 4** là cùng bài toán ở dạng "giữ lại tối đa": `w[i][j]` là LCS có trọng số chuẩn (cũng được chứng minh bằng đúng lập luận khớp ký tự cuối ở trên).

## 11. Phân tích Time / Space Complexity

`m = len(s1)`, `n = len(s2)` (mỗi cái tối đa `1000`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đệ quy không nhớ | `O(2^(m+n))` | `O(m + n)` | chỉ cho chuỗi rất ngắn |
| 2. Đệ quy có nhớ | `O(m*n)` | `O(m*n)` + ngăn xếp `O(m+n)` | cần nâng `recursionlimit` |
| 3. Bảng DP | `O(m*n)` | `O(m*n)` | **dùng để nộp** |
| 4. LCS trọng số | `O(m*n)` | `O(m*n)` | góc nhìn khác, dễ truy vết chuỗi giữ lại |
| 5. DP một hàng | `O(m*n)` | `O(n)` | tiết kiệm bộ nhớ |

Với `m = n = 1000` là `10^6` ô, mỗi ô vài phép so sánh: chạy trong khoảng một giây trong Python. Giá trị lớn nhất của đáp án: `2000 * 122 = 244.000`, nằm gọn trong số nguyên thường.

## 12. Bộ test

### 12.1. Ví dụ của đề và oracle độc lập

Oracle độc lập: **liệt kê mọi dãy con của `s1`** (bitmask), giữ những dãy cũng là dãy con của `s2`, lấy dãy có tổng ASCII lớn nhất; đáp án `= tổng - 2 * giữ`. Cách này không dùng bảng DP (chỉ cho chuỗi ngắn, `m <= 12`).

In [23]:
def oracle_subseq(s1, s2):
    best = 0
    m = len(s1)
    for mask in range(1 << m):
        t = "".join(s1[i] for i in range(m) if mask >> i & 1)
        w = sum(map(ord, t))
        if w > best and is_subsequence(t, s2):
            best = w
    return sum(map(ord, s1)) + sum(map(ord, s2)) - 2 * best

METHODS = {
    "1-de-quy": Solution1().minimumDeleteSum,
    "2-co-nho": Solution2().minimumDeleteSum,
    "3-bang-dp": Solution3().minimumDeleteSum,
    "4-lcs-trong-so": Solution4().minimumDeleteSum,
    "5-mot-hang": Solution5().minimumDeleteSum,
    "nop": Solution().minimumDeleteSum,
}

def check_all(s1, s2, expected=None, skip=()):
    exp = oracle_subseq(s1, s2) if expected is None else expected
    for name, fn in METHODS.items():
        if name in skip:
            continue
        got = fn(s1, s2)
        assert got == exp, (name, s1, s2, got, exp)
    return exp

for (a, b), expected in examples:
    assert check_all(a, b, expected) == oracle_subseq(a, b) == expected
print("ví dụ của đề: OK")

ví dụ của đề: OK


### 12.2. Edge case

- Hai chuỗi **giống hệt**: không xóa gì, kết quả `0`.
- Hai chuỗi **không có chữ chung**: phải xóa hết cả hai (tổng ASCII cả hai).
- Một chuỗi là **dãy con** của chuỗi kia: xóa phần dư của chuỗi dài.
- Chuỗi độ dài `1`; chuỗi một chữ lặp lại; chuỗi đảo ngược.
- Cùng chữ cái nhưng khác thứ tự: `"ab"` và `"ba"`.

In [24]:
edge = [
    ("a", "a", 0),
    ("a", "b", 97 + 98),
    ("abc", "abc", 0),
    ("abc", "xyz", sum(map(ord, "abcxyz"))),
    ("abc", "ac", 98),                       # xóa b
    ("ac", "abc", 98),
    ("aaaa", "aa", 2 * 97),
    ("ab", "ba", None),                      # kiểm bằng oracle
    ("abcd", "dcba", None),
    ("zzz", "z", 2 * 122),
    ("leetcode", "etco", None),
]
for a, b, expected in edge:
    exp = check_all(a, b, expected) if expected is not None else check_all(a, b)
    print(f"{a!r:12} {b!r:8} -> {exp}")
print("edge case: OK")

'a'          'a'      -> 0
'a'          'b'      -> 195
'abc'        'abc'    -> 0
'abc'        'xyz'    -> 657
'abc'        'ac'     -> 98
'ac'         'abc'    -> 98
'aaaa'       'aa'     -> 194
'ab'         'ba'     -> 194
'abcd'       'dcba'   -> 588
'zzz'        'z'      -> 244
'leetcode'   'etco'   -> 410
edge case: OK


### 12.3. Phản ví dụ: input làm lộ các lỗi quen thuộc

- Dãy con chung **dài nhất** (theo số ký tự) không nhất thiết có **ASCII lớn nhất**: `"ab"` và `"ba"`; giữ `"a"` hay `"b"` có ASCII khác nhau (xem bài tập 2).
- Tham lam "xóa mọi ký tự không có ở chuỗi kia" cho chuỗi không bằng nhau: `"delete"`/`"leet"` (mục 13).
- Một số cặp mà đáp án **lệch** khi quên cộng ASCII: kiểm ở mục 13.

In [25]:
tricky = [
    ("ab", "ba"),             # giữ a hay giữ b: ASCII khác nhau
    ("azb", "bza"),           # giữ z (122) hay a/b?
    ("aaab", "baaa"),
    ("xyz", "zyx"),
    ("abab", "baba"),
    ("zaz", "azz"),
]
for a, b in tricky:
    print(f"{a!r:7} {b!r:7} -> {check_all(a, b)}")

'ab'    'ba'    -> 194
'azb'   'bza'   -> 390
'aaab'  'baaa'  -> 196
'xyz'   'zyx'   -> 482
'abab'  'baba'  -> 194
'zaz'   'azz'   -> 194


### 12.4. Đối chiếu ngẫu nhiên (seed cố định) với oracle vét cạn

Chuỗi ngẫu nhiên độ dài `1..9` trên bảng chữ nhỏ (`"ab"`, `"abc"`, `"abcd"`, hoặc cả bảng chữ cái) để có nhiều khớp. Cả sáu cách phải bằng oracle.

In [26]:
rng = random.Random(712)
count = 0
for _ in range(600):
    alphabet = rng.choice(["ab", "abc", "abcd", "abcdefghijklmnopqrstuvwxyz"])
    a = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 8)))
    b = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 8)))
    check_all(a, b)
    count += 1
print("đã đối chiếu", count, "cặp ngẫu nhiên với", len(METHODS), "cách: OK")

đã đối chiếu 600 cặp ngẫu nhiên với 6 cách: OK


Thêm lượt **chuỗi dài hơn** (độ dài `20..60`) cho các cách đa thức: đối chiếu chéo giữa các cách (bỏ cách 1 vì hàm mũ) và so sánh với cách 3. Đây là kiểm chứng "nhất quán" (hai góc nhìn DP khác nhau phải ra cùng đáp án).

In [27]:
rng = random.Random(7120)
for _ in range(300):
    alphabet = rng.choice(["ab", "abc", "abcdefghij", "abcdefghijklmnopqrstuvwxyz"])
    a = "".join(rng.choice(alphabet) for _ in range(rng.randint(20, 60)))
    b = "".join(rng.choice(alphabet) for _ in range(rng.randint(20, 60)))
    ref = METHODS["3-bang-dp"](a, b)
    for name in ["2-co-nho", "4-lcs-trong-so", "5-mot-hang", "nop"]:
        assert METHODS[name](a, b) == ref, (name, a, b)
    # tính chất: đối xứng theo hai chuỗi, và không vượt tổng ASCII, không âm
    assert ref == METHODS["3-bang-dp"](b, a)
    assert 0 <= ref <= sum(map(ord, a + b))
print("chuỗi dài: nhất quán + đối xứng: OK")

chuỗi dài: nhất quán + đối xứng: OK


### 12.5. Hiệu năng ở giới hạn (`1000 x 1000`)

Hai chuỗi ngẫu nhiên dài `1000`: cách 3, 4, 5 phải chạy trong khoảng một giây. Cách 2 (đệ quy sâu) thử ở kích thước `300` để tránh nặng bộ nhớ; cách 1 không thử.

In [28]:
rng = random.Random(1000)
a = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(1000))
b = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(1000))
results = {}
for name in ["3-bang-dp", "4-lcs-trong-so", "5-mot-hang"]:
    t0 = time.perf_counter()
    results[name] = METHODS[name](a, b)
    print(f"{name:15} -> {results[name]} trong {(time.perf_counter() - t0) * 1000:7.1f} ms")
assert len(set(results.values())) == 1

a2, b2 = a[:300], b[:300]
t0 = time.perf_counter()
r2 = METHODS["2-co-nho"](a2, b2)
print(f"{'2-co-nho (300)':15} -> {r2} trong {(time.perf_counter() - t0) * 1000:7.1f} ms")
assert r2 == METHODS["3-bang-dp"](a2, b2)

# trường hợp đặc biệt: chuỗi giống hệt và chuỗi không có chữ chung, độ dài 1000
assert METHODS["nop"]("a" * 1000, "a" * 1000) == 0
assert METHODS["nop"]("a" * 1000, "b" * 1000) == 1000 * (97 + 98)
print("giống hệt -> 0, không chữ chung -> 1000*(97+98): OK")

3-bang-dp       -> 150692 trong   244.9 ms
4-lcs-trong-so  -> 150692 trong   148.4 ms


5-mot-hang      -> 150692 trong   206.7 ms
2-co-nho (300)  -> 46454 trong    48.5 ms


giống hệt -> 0, không chữ chung -> 1000*(97+98): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm số ký tự xóa thay vì tổng ASCII (giải bài LCS/Delete Operation)

**Code sai:** mỗi lần xóa cộng `1` thay vì `ord(...)`.

In [29]:
def wrong_count_chars(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for j in range(1, n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        dp[i][0] = i
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(dp[i - 1][j] + 1, dp[i][j - 1] + 1)      # SAI: cộng 1
    return dp[m][n]

print("sai  :", wrong_count_chars("sea", "eat"), "(số ký tự bị xóa)")
print("đúng :", Solution().minimumDeleteSum("sea", "eat"))
print("với 'ab','ba':", wrong_count_chars("ab", "ba"), "(mọi ký tự đều giá 1) so với đúng:", Solution().minimumDeleteSum("ab", "ba"))

sai  : 2 (số ký tự bị xóa)
đúng : 231
với 'ab','ba': 2 (mọi ký tự đều giá 1) so với đúng: 194


**Vì sao sai:** bài này là LeetCode 583 (đếm số lần xóa) nhưng đề 712 tính **tổng mã ASCII**. Hai ký tự khác nhau có chi phí khác nhau, nên việc chọn xóa ký tự nào quan trọng (ví dụ `"ab"` và `"ba"`: xóa hai ký tự `a` hay hai ký tự `b` đều xóa 2 ký tự nhưng ASCII khác nhau).

**Code đúng:** cộng `ord(ký tự bị xóa)`.

### Lỗi 2: Điều kiện biên sai (hàng/cột 0 để bằng `0`)

In [30]:
def wrong_base(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]           # SAI: không điền tổng tiền tố cho hàng/cột 0
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(dp[i - 1][j] + ord(s1[i - 1]), dp[i][j - 1] + ord(s2[j - 1]))
    return dp[m][n]

print("sai  :", wrong_base("sea", "eat"), "(thấp hơn: coi việc xóa hết tiền tố là miễn phí)")
print("đúng :", Solution().minimumDeleteSum("sea", "eat"))

sai  : 116 (thấp hơn: coi việc xóa hết tiền tố là miễn phí)
đúng : 231


**Vì sao sai:** `dp[0][j]` phải là `ASCII(s2[:j])` (xóa hết `j` ký tự). Để `0` nghĩa là xóa miễn phí, đáp án nhỏ hơn thực tế.

**Code đúng:** điền hàng 0 và cột 0 bằng tổng tiền tố trước khi chạy vòng lặp chính.

### Lỗi 3: Lệch chỉ số (dùng `s1[i]` thay vì `s1[i-1]`)

In [31]:
def wrong_index(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for j in range(1, n + 1):
        dp[0][j] = dp[0][j - 1] + ord(s2[j - 1])
    for i in range(1, m + 1):
        dp[i][0] = dp[i - 1][0] + ord(s1[i - 1])
        for j in range(1, n + 1):
            if s1[i] == s2[j]:                           # SAI: dp dùng độ dài tiền tố, ký tự cuối là s1[i-1]
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(dp[i - 1][j] + ord(s1[i]), dp[i][j - 1] + ord(s2[j]))
    return dp[m][n]

try:
    wrong_index("sea", "eat")
except IndexError as e:
    print("sai  : IndexError:", e)
print("đúng :", Solution().minimumDeleteSum("sea", "eat"))

sai  : IndexError: string index out of range
đúng : 231


**Vì sao sai:** `dp[i][j]` ứng với **`i` ký tự đầu**, nên ký tự cuối của tiền tố là `s1[i-1]`. Dùng `s1[i]` trượt một ký tự và vượt biên ở `i = m`.

**Code đúng:** luôn dùng `s1[i - 1]`, `s2[j - 1]`.

### Lỗi 4: Tham lam "xóa mọi ký tự không có ở chuỗi kia"

In [32]:
def wrong_greedy(s1, s2):
    cost = sum(ord(c) for c in s1 if c not in s2) + sum(ord(c) for c in s2 if c not in s1)
    return cost                                          # SAI: các ký tự còn lại chưa chắc đã bằng nhau

for a, b in [("sea", "eat"), ("delete", "leet"), ("ab", "ba"), ("abc", "cba")]:
    print(f"{a!r:9} {b!r:7} tham lam = {wrong_greedy(a, b):>4} | đúng = {Solution().minimumDeleteSum(a, b)}")

'sea'     'eat'   tham lam =  231 | đúng = 231
'delete'  'leet'  tham lam =  100 | đúng = 403
'ab'      'ba'    tham lam =    0 | đúng = 194
'abc'     'cba'   tham lam =    0 | đúng = 390


**Vì sao sai:** sau khi xóa các ký tự "vắng mặt", hai chuỗi còn lại có thể vẫn **khác thứ tự hoặc khác số lượng** (ví dụ `"delete"` chỉ mất `d` thành `"elete"` còn `"leet"` giữ nguyên: hai chuỗi vẫn khác nhau). Phải chọn một **dãy con chung** thực sự, đó là nguồn gốc của quy hoạch động.

**Code đúng:** DP (cách 3/4).

### Lỗi 5: Tạo bảng bằng `[[0] * (n + 1)] * (m + 1)`

In [33]:
def wrong_alias(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1)] * (m + 1)                      # SAI: mọi hàng là cùng một list
    for j in range(1, n + 1):
        dp[0][j] = dp[0][j - 1] + ord(s2[j - 1])
    for i in range(1, m + 1):
        dp[i][0] = dp[i - 1][0] + ord(s1[i - 1])
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(dp[i - 1][j] + ord(s1[i - 1]), dp[i][j - 1] + ord(s2[j - 1]))
    return dp[m][n]

print("sai  :", wrong_alias("sea", "eat"))
print("đúng :", Solution().minimumDeleteSum("sea", "eat"))

sai  : 429
đúng : 231


**Vì sao sai:** (mục 3.4) mọi hàng cùng một list nên ghi `dp[i][j]` làm đổi **mọi hàng**; các giá trị "hàng trước" bị ghi đè.

**Code đúng:** `[[0] * (n + 1) for _ in range(m + 1)]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nhận dạng:** "xóa ký tự để hai chuỗi bằng nhau với chi phí" = **LCS (dãy con chung)** hoặc **edit distance chỉ có phép xóa**. Hai chuỗi, tiền tố `i`, `j` -> bảng DP 2 chiều.
- **Mẫu bất biến:** ký tự cuối bằng nhau -> chéo; khác nhau -> thử bỏ ký tự cuối của từng chuỗi (hai nhánh) và lấy tốt nhất.
- **Đổi góc nhìn:** "tối thiểu hóa phần bị xóa" = "tối đa hóa phần giữ lại"; phần giữ lại là dãy con chung có trọng số lớn nhất.
- **Điều kiện biên** = chi phí xóa hết một tiền tố (tổng dồn), đừng để `0`.
- **Tránh lệch chỉ số:** dùng `dp` kích thước `(m+1) x (n+1)` và ký tự `s[i-1]`.
- **Tối ưu bộ nhớ:** mỗi hàng chỉ cần hàng trước -> mảng một hàng (nhớ `diag`).
- **Kiểm tra:** thử chuỗi giống hệt (`0`), không chữ chung (tổng ASCII cả hai), một chuỗi là dãy con của chuỗi kia.
- Bài họ hàng: LeetCode 1143 (Longest Common Subsequence), 583 (Delete Operation for Two Strings), 72 (Edit Distance), 1092 (Shortest Common Supersequence), 97 (Interleaving String), 115 (Distinct Subsequences), 516 (Longest Palindromic Subsequence).

## 15. Biến thể

### 15.1. Truy vết: in ra chuỗi được giữ lại và các ký tự bị xóa

Dùng bảng `dp` đầy đủ (cách 3), đi ngược từ `(m, n)` về `(0, 0)`: khớp thì giữ ký tự; ngược lại đi theo ô đã cho giá trị nhỏ.

In [34]:
def with_plan(s1, s2):
    dp = dp_table(s1, s2)
    i, j = len(s1), len(s2)
    kept, del1, del2 = [], [], []
    while i > 0 and j > 0:
        if s1[i - 1] == s2[j - 1]:
            kept.append(s1[i - 1]); i -= 1; j -= 1
        elif dp[i][j] == dp[i - 1][j] + ord(s1[i - 1]):
            del1.append(s1[i - 1]); i -= 1
        else:
            del2.append(s2[j - 1]); j -= 1
    del1 += list(reversed(s1[:i]))
    del2 += list(reversed(s2[:j]))
    return "".join(reversed(kept)), "".join(reversed(del1)), "".join(reversed(del2))

for a, b in [("sea", "eat"), ("delete", "leet")]:
    kept, d1, d2 = with_plan(a, b)
    cost = sum(map(ord, d1 + d2))
    print(f"{a!r},{b!r}: giữ {kept!r}, xóa khỏi s1 {d1!r}, xóa khỏi s2 {d2!r}, chi phí {cost}")
    assert cost == Solution().minimumDeleteSum(a, b)
    assert is_subsequence(kept, a) and is_subsequence(kept, b)

'sea','eat': giữ 'ea', xóa khỏi s1 's', xóa khỏi s2 't', chi phí 231
'delete','leet': giữ 'let', xóa khỏi s1 'dee', xóa khỏi s2 'e', chi phí 403


### 15.2. LeetCode 583: chỉ đếm số lần xóa (mỗi ký tự giá `1`)

Đổi chi phí `ord(c)` thành `1`: đáp án `= m + n - 2 * LCS`. Có thể dùng chính bảng `w` của cách 4 với trọng số `1`.

In [35]:
def min_delete_count(s1, s2):
    m, n = len(s1), len(s2)
    w = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                w[i][j] = w[i - 1][j - 1] + 1
            else:
                w[i][j] = max(w[i - 1][j], w[i][j - 1])
    return m + n - 2 * w[m][n]

print(min_delete_count("sea", "eat"), min_delete_count("leetcode", "etco"))

2 4


### 15.3. Thêm phép thay thế (edit distance có trọng số)

Cho phép **thay** một ký tự bằng ký tự khác với chi phí `|ord(a) - ord(b)|`, ngoài việc xóa/chèn với chi phí `ord`. Ta thêm nhánh chéo `dp[i-1][j-1] + abs(ord(a) - ord(b))`. Đây là họ hàng gần của bài này.

In [36]:
def weighted_edit(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for j in range(1, n + 1):
        dp[0][j] = dp[0][j - 1] + ord(s2[j - 1])
    for i in range(1, m + 1):
        dp[i][0] = dp[i - 1][0] + ord(s1[i - 1])
        for j in range(1, n + 1):
            a, b = s1[i - 1], s2[j - 1]
            dp[i][j] = min(dp[i - 1][j] + ord(a), dp[i][j - 1] + ord(b), dp[i - 1][j - 1] + abs(ord(a) - ord(b)))
    return dp[m][n]

print(weighted_edit("sea", "eat"), "<=", Solution().minimumDeleteSum("sea", "eat"))
assert all(weighted_edit(a, b) <= Solution().minimumDeleteSum(a, b) for a, b in [("sea", "eat"), ("ab", "ba"), ("abc", "xyz")])

37 <= 231


### 15.4. Dãy con chung ngắn nhất chứa cả hai (Shortest Common Supersequence)

Độ dài `m + n - LCS`. Biến thể này chỉ cần LCS không trọng số (`15.2`), nhưng để **dựng chuỗi** ta dùng truy vết như 15.1 (LeetCode 1092).

In [37]:
def scs_length(s1, s2):
    m, n = len(s1), len(s2)
    lcs = (m + n - min_delete_count(s1, s2)) // 2
    return m + n - lcs

print(scs_length("abac", "cab"), scs_length("sea", "eat"))

5 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy không nhớ | thử mọi cách xóa theo hậu tố | `O(2^(m+n))` | `O(m+n)` | chỉ để hiểu cấu trúc |
| 2. Đệ quy có nhớ | `f(i, j)` + `lru_cache` | `O(m*n)` | `O(m*n)` | viết nhanh, dễ suy ra từ cách 1 |
| 3. Bảng DP | `dp[i][j]` tối thiểu hóa chi phí xóa | `O(m*n)` | `O(m*n)` | **nộp bài** |
| 4. LCS trọng số | tối đa hóa ASCII giữ lại, đáp án `tổng - 2w` | `O(m*n)` | `O(m*n)` | cần truy vết/giải thích "giữ gì" |
| 5. DP một hàng | cuộn mảng, nhớ ô chéo | `O(m*n)` | `O(n)` | giới hạn bộ nhớ |

Kết luận: nộp **cách 3**. Cách 4 là cùng bài toán ở dạng đối ngẫu, rất tiện đối chiếu; cách 5 khi muốn tiết kiệm bộ nhớ.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán đầu ra rồi chạy:

In [38]:
print(Solution().minimumDeleteSum("a", "a"))
print(Solution().minimumDeleteSum("ab", "b"))
print(Solution().minimumDeleteSum("ab", "cd"))

0
97
394


**Bài 2 (tính tay).** `s1 = "ab"`, `s2 = "ba"`. Liệt kê các dãy con chung (`""`, `"a"`, `"b"`) và chi phí xóa tương ứng; gán chi phí **nhỏ nhất** vào `ans2` (đang là `None`).

In [39]:
ans2 = None      # TODO: điền một số nguyên

**Bài 3 (điền TODO).** Viết `prefix_ascii(s)` trả về list độ dài `len(s) + 1`, trong đó phần tử `i` là tổng ASCII của `s[:i]`. Ví dụ `prefix_ascii("sea")` -> `[0, 115, 216, 313]`.

In [40]:
def prefix_ascii(s):
    # TODO: dùng accumulate hoặc vòng lặp
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết `my_min_delete(s1, s2)` theo cách 3 (bảng DP, có điền hàng/cột 0).

In [41]:
def my_min_delete(s1, s2):
    # TODO
    pass

**Bài 5 (mini challenge).** (a) Viết `kept_string(s1, s2)` trả về **một** chuỗi được giữ lại ở một phương án tối ưu (dãy con chung có tổng ASCII lớn nhất). (b) Viết `delete_count_and_sum(s1, s2)` trả về cặp `(số ký tự bị xóa nhỏ nhất, tổng ASCII nhỏ nhất)` **riêng biệt** cho hai bài toán (không cần cùng một phương án).

In [42]:
def kept_string(s1, s2):
    # TODO: truy vết từ bảng DP (xem mục 15.1)
    pass

def delete_count_and_sum(s1, s2):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: dòng 1 hai chuỗi bằng nhau; dòng 2 chỉ cần xóa `a`; dòng 3 không có chữ chung.
- Bài 2: nếu giữ `"a"`, phải xóa `b` ở cả hai chuỗi; nếu giữ `"b"`, phải xóa `a` ở cả hai.
- Bài 3: `accumulate(map(ord, s))` cho tổng dồn (không có số 0 đầu).
- Bài 4: bảng `(m+1) x (n+1)`, điền biên trước, ký tự cuối là `s[i-1]`.
- Bài 5: tận dụng bảng và truy vết ở mục 15.1.

**Mức 2 (vừa):**
- Bài 2: chi phí giữ `"a"`: `98 + 98 = 196`; giữ `"b"`: `97 + 97 = 194`; không giữ gì: `97 + 98 + 98 + 97`.
- Bài 3: `[0] + list(accumulate(map(ord, s)))`.
- Bài 4: `if s1[i-1] == s2[j-1]: dp[i-1][j-1] else min(dp[i-1][j] + ord(s1[i-1]), dp[i][j-1] + ord(s2[j-1]))`.
- Bài 5(b): hai bảng riêng: một bảng với chi phí `ord`, một bảng với chi phí `1` (mục 15.2).

**Mức 3 (gần đáp án):**
- Bài 2: nhỏ nhất là `194` (giữ `"b"`).
- Bài 3: `return [0] + list(accumulate(map(ord, s)))`.
- Bài 5(a): đi ngược từ `(m, n)`: nếu `s1[i-1] == s2[j-1]` thì thêm ký tự đó vào kết quả rồi `i -= 1; j -= 1`; nếu `dp[i][j] == dp[i-1][j] + ord(s1[i-1])` thì `i -= 1`; ngược lại `j -= 1`; cuối cùng đảo ngược chuỗi kết quả.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [43]:
# Bài 1: 0, 97, 97+98+99+100 = 394
assert Solution().minimumDeleteSum("a", "a") == 0
assert Solution().minimumDeleteSum("ab", "b") == 97
assert Solution().minimumDeleteSum("ab", "cd") == 97 + 98 + 99 + 100

# Bài 2: giữ "b": xóa a ở cả hai = 97 + 97 = 194
ans2 = 194
assert ans2 == Solution().minimumDeleteSum("ab", "ba") == oracle_subseq("ab", "ba")
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [44]:
# Bài 3
def prefix_ascii(s):
    return [0] + list(accumulate(map(ord, s)))

assert prefix_ascii("sea") == [0, 115, 216, 313]
assert prefix_ascii("a") == [0, 97]
assert prefix_ascii("abc")[2] == sum(map(ord, "ab"))
print("Bài 3 OK")

Bài 3 OK


In [45]:
# Bài 4
def my_min_delete(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for j in range(1, n + 1):
        dp[0][j] = dp[0][j - 1] + ord(s2[j - 1])
    for i in range(1, m + 1):
        dp[i][0] = dp[i - 1][0] + ord(s1[i - 1])
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = min(dp[i - 1][j] + ord(s1[i - 1]), dp[i][j - 1] + ord(s2[j - 1]))
    return dp[m][n]

rng = random.Random(4)
for _ in range(200):
    a = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    b = "".join(rng.choice("abc") for _ in range(rng.randint(1, 8)))
    assert my_min_delete(a, b) == oracle_subseq(a, b)
print("Bài 4 OK")

Bài 4 OK


In [46]:
# Bài 5
def kept_string(s1, s2):
    dp = dp_table(s1, s2)
    i, j = len(s1), len(s2)
    out = []
    while i > 0 and j > 0:
        if s1[i - 1] == s2[j - 1]:
            out.append(s1[i - 1])
            i -= 1
            j -= 1
        elif dp[i][j] == dp[i - 1][j] + ord(s1[i - 1]):
            i -= 1
        else:
            j -= 1
    return "".join(reversed(out))

def delete_count_and_sum(s1, s2):
    return min_delete_count(s1, s2), Solution().minimumDeleteSum(s1, s2)

assert kept_string("sea", "eat") == "ea"
assert kept_string("delete", "leet") == "let"
assert delete_count_and_sum("sea", "eat") == (2, 231)
rng = random.Random(5)
for _ in range(200):
    a = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 8)))
    b = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 8)))
    t = kept_string(a, b)
    assert is_subsequence(t, a) and is_subsequence(t, b)
    assert sum(map(ord, a + b)) - 2 * sum(map(ord, t)) == oracle_subseq(a, b)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template bảng DP (tối thiểu hóa tổng ASCII bị xóa):**

```python
class Solution:
    def minimumDeleteSum(self, s1: str, s2: str) -> int:
        m, n = len(s1), len(s2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for j in range(1, n + 1):
            dp[0][j] = dp[0][j - 1] + ord(s2[j - 1])      # xóa hết s2[:j]
        for i in range(1, m + 1):
            dp[i][0] = dp[i - 1][0] + ord(s1[i - 1])      # xóa hết s1[:i]
            for j in range(1, n + 1):
                if s1[i - 1] == s2[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1]           # khớp: miễn phí
                else:
                    dp[i][j] = min(dp[i - 1][j] + ord(s1[i - 1]),
                                   dp[i][j - 1] + ord(s2[j - 1]))
        return dp[m][n]
```

**Đối ngẫu (LCS trọng số):** `đáp án = sum(ord) của cả hai - 2 * w[m][n]`, với `w[i][j] = w[i-1][j-1] + ord(c)` khi khớp, ngược lại `max(w[i-1][j], w[i][j-1])`.

**Checklist trước khi submit:**

- [ ] Cộng **`ord(ký tự)`**, không phải `1`?
- [ ] Hàng 0 và cột 0 là **tổng ASCII của tiền tố** (không phải `0`)?
- [ ] Dùng `s1[i - 1]`, `s2[j - 1]` (bảng có kích thước `(m+1) x (n+1)`)?
- [ ] Hai ký tự bằng nhau -> lấy ô **chéo** (không cộng gì)?
- [ ] Bảng tạo bằng list comprehension, **không** `[[0]*n]*m`?
- [ ] Đã thử: hai chuỗi giống hệt (`0`), không chữ chung, một chuỗi là dãy con của chuỗi kia, độ dài `1`?
- [ ] Nếu dùng đệ quy có nhớ: đã nâng `recursionlimit` cho `m + n` tới `2000`?